# SentimentLab AI (P4): Hugging Face BERT Fine-Tuning
### Academic Mini-Project: Generative vs Discriminative Sentiment Analysis Laboratory

This Google Colab notebook provides an end-to-end, reproducible pipeline to fine-tune **`bert-base-uncased`** on an NVIDIA T4 GPU for 3-class sentiment analysis (`0=Negative`, `1=Neutral`, `2=Positive`).

**GPU Setup:**
- Menu: `Runtime` -> `Change runtime type` -> Hardware accelerator: `T4 GPU`.

In [ ]:
# Step 1: Verify Hardware Accelerator
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: Running on CPU. Please switch to T4 GPU runtime!")

In [ ]:
# Step 2: Install Dependencies
!pip install -q transformers datasets accelerate scikit-learn pandas matplotlib seaborn pyarrow

In [ ]:
# Step 3: Load Dataset Splits
# You can either upload your local Parquet files (train_real.parquet, val_real.parquet, eval_subset_real.parquet)
# or let this cell load and stratify directly from Hugging Face yelp_review_full.

import os
import pandas as pd
import numpy as np

if os.path.exists("train_real.parquet") and os.path.exists("val_real.parquet") and os.path.exists("eval_subset_real.parquet"):
    print("Loading uploaded local project Parquet files...")
    train_subset = pd.read_parquet("train_real.parquet")
    val_subset = pd.read_parquet("val_real.parquet")
    eval_subset = pd.read_parquet("eval_subset_real.parquet")
    text_col = "review"
    label_col = "label"
else:
    print("Downloading and stratifying Yelp Review Full from Hugging Face...")
    from datasets import load_dataset
    dataset = load_dataset("yelp_review_full")
    
    def map_stars_to_3class(example):
        lbl = example["label"]
        if lbl in (0, 1):
            example["label"] = 0
        elif lbl == 2:
            example["label"] = 1
        else:
            example["label"] = 2
        return example
        
    dataset = dataset.map(map_stars_to_3class)
    train_df = pd.DataFrame(dataset["train"])
    test_df = pd.DataFrame(dataset["test"])
    
    def stratified_sample(df, n_per_class, seed=42):
        return df.groupby("label", group_keys=False).apply(
            lambda x: x.sample(n=n_per_class, random_state=seed)
        ).sample(frac=1.0, random_state=seed).reset_index(drop=True)
        
    train_subset = stratified_sample(train_df, 6400, seed=42)
    rem_train = train_df.drop(train_subset.index, errors="ignore")
    val_subset = stratified_sample(rem_train, 1600, seed=42)
    eval_subset = stratified_sample(test_df, 1000, seed=42)
    text_col = "text"
    label_col = "label"

print(f"Dataset ready: Train={len(train_subset)}, Val={len(val_subset)}, Eval={len(eval_subset)}")

In [ ]:
# Step 4: Tokenization & PyTorch DataLoaders
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_ds = SentimentDataset(train_subset[text_col].tolist(), train_subset[label_col].tolist())
val_ds = SentimentDataset(val_subset[text_col].tolist(), val_subset[label_col].tolist())
eval_ds = SentimentDataset(eval_subset[text_col].tolist(), eval_subset[label_col].tolist())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False)
print("DataLoaders prepared successfully.")

In [ ]:
# Step 5: Fine-Tune BERT (AdamW, Warmup Scheduler, Early Stopping)
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: "Negative", 1: "Neutral", 2: "Positive"},
    label2id={"Negative": 0, "Neutral": 1, "Positive": 2}
)
model.to(device)

EPOCHS = 3
LR = 2e-5
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}
best_val_loss = float("inf")
os.makedirs("bert_checkpoint", exist_ok=True)

train_start_time = time.time()
print(f"Training on {device} for {EPOCHS} epochs...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    preds, targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            p = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            preds.extend(p)
            targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    macro_f1 = f1_score(targets, preds, average="macro")
    history["train_loss"].append(float(avg_train_loss))
    history["val_loss"].append(float(avg_val_loss))
    history["val_macro_f1"].append(float(macro_f1))

    print(f"Epoch {epoch}/{EPOCHS} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro-F1: {macro_f1:.4f}")
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained("bert_checkpoint")
        tokenizer.save_pretrained("bert_checkpoint")
        print(f"Saved best model checkpoint to bert_checkpoint/ (val_loss: {avg_val_loss:.4f})")

training_time = time.time() - train_start_time
print(f"Fine-tuning completed in {training_time:.2f} seconds.")

In [ ]:
# Step 6: Comprehensive Benchmark Evaluation & Publication Charts
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, log_loss

model = AutoModelForSequenceClassification.from_pretrained("bert_checkpoint").to(device)
model.eval()

eval_preds, eval_probs, eval_targets = [], [], []
start_eval = time.time()
with torch.no_grad():
    for batch in eval_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        p = np.argmax(probs, axis=1)
        eval_probs.extend(probs)
        eval_preds.extend(p)
        eval_targets.extend(labels.cpu().numpy())

eval_duration = time.time() - start_eval
eval_probs_arr = np.array(eval_probs)
# Normalize row sums
eval_probs_arr = eval_probs_arr / np.sum(eval_probs_arr, axis=1, keepdims=True)

acc = float(accuracy_score(eval_targets, eval_preds))
macro_f1 = float(f1_score(eval_targets, eval_preds, average="macro"))
cm = confusion_matrix(eval_targets, eval_preds).tolist()
clf_report = classification_report(eval_targets, eval_preds, target_names=["Negative", "Neutral", "Positive"], output_dict=True)

# Multiclass Brier Score
y_true_onehot = np.eye(3)[eval_targets]
brier_score = float(np.mean(np.sum((eval_probs_arr - y_true_onehot) ** 2, axis=1)))
loss_val = float(log_loss(eval_targets, eval_probs_arr, labels=[0, 1, 2]))

eval_metrics = {
    "accuracy": acc,
    "macro_f1": macro_f1,
    "brier_score": brier_score,
    "log_loss": loss_val,
    "per_class": {
        "Negative": clf_report["Negative"],
        "Neutral": clf_report["Neutral"],
        "Positive": clf_report["Positive"],
    },
    "confusion_matrix": cm,
    "inference_latency_per_sample_ms": float((eval_duration / len(eval_subset)) * 1000.0),
}

with open("bert_checkpoint/evaluation_metrics.json", "w") as f:
    json.dump(eval_metrics, f, indent=2)
with open("bert_checkpoint/training_history.json", "w") as f:
    json.dump(history, f, indent=2)

# Model Metadata
metadata = {
    "model_name": "bert-base-uncased",
    "version": "1.0.0",
    "device": str(device),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "max_length": MAX_LEN,
    "training_duration_seconds": round(training_time, 2),
}
with open("bert_checkpoint/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Save Predictions Parquet
pred_df = eval_subset.copy()
pred_df["predicted_label"] = eval_preds
pred_df["predicted_sentiment"] = [["Negative", "Neutral", "Positive"][l] for l in eval_preds]
pred_df["confidence"] = np.max(eval_probs_arr, axis=1)
pred_df["prob_negative"] = eval_probs_arr[:, 0]
pred_df["prob_neutral"] = eval_probs_arr[:, 1]
pred_df["prob_positive"] = eval_probs_arr[:, 2]
pred_df.to_parquet("bert_checkpoint/predictions.parquet")

# Generate Charts
plt.figure(figsize=(6, 4.5), dpi=300)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Negative", "Neutral", "Positive"], yticklabels=["Negative", "Neutral", "Positive"])
plt.title(f"BERT Confusion Matrix (n={len(eval_subset)})", fontweight="bold")
plt.savefig("bert_checkpoint/confusion_matrix.png", bbox_inches="tight")
plt.close()

plt.figure(figsize=(7, 4.5), dpi=300)
epochs_range = range(1, len(history["train_loss"]) + 1)
plt.plot(epochs_range, history["train_loss"], 'o-', label="Train Loss", color="red")
plt.plot(epochs_range, history["val_loss"], 's--', label="Val Loss", color="blue")
plt.title("BERT Loss Curve", fontweight="bold")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.savefig("bert_checkpoint/loss_curve.png", bbox_inches="tight")
plt.close()

print(f"Evaluation Complete: Accuracy={acc:.4f} | Macro-F1={macro_f1:.4f} | Brier={brier_score:.4f}")

In [ ]:
# Step 7: Export All Artifacts for SentimentLab AI
!zip -r bert_artifacts.zip bert_checkpoint/
from google.colab import files
files.download("bert_artifacts.zip")
print("Artifacts downloaded successfully as bert_artifacts.zip!")
print("Extraction instructions:")
print("  1. Extract model files (model.safetensors, config.json, tokenizer.json, etc.) into artifacts/models/bert/")
print("  2. Extract results files (evaluation_metrics.json, training_history.json, confusion_matrix.png, etc.) into artifacts/results/bert/")